In [ ]:
import duckdb
from pathlib import Path
from datetime import date

In [ ]:
con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

silver_files = sorted(Path("../../data/silver").glob("cleaned_data_load_date=*.parquet"))
if not silver_files:
    raise FileNotFoundError("No Silver Parquet found.")

silver_path = silver_files[-1].as_posix()
con.execute(f"""
    CREATE OR REPLACE VIEW silver_data AS
    SELECT * FROM read_parquet('{silver_path}')
""")
print(f"Silver: {silver_files[-1].name}")
print("Rows:", con.execute("SELECT COUNT(*) FROM silver_data").fetchone()[0])

In [ ]:
con.execute("DROP TABLE IF EXISTS gold.review_product_context")
con.execute("""
    CREATE TABLE gold.review_product_context AS
    WITH base AS (
        SELECT
            _index,
            product_id,
            product_parent,
            label,
            vine,
            verified_purchase,
            review_date,
            marketplace_id,
            product_category_id,

            -- Recompute temporal features needed for interactions
            COUNT(*) OVER (PARTITION BY product_parent)           AS product_review_count,
            ROW_NUMBER() OVER (
                PARTITION BY product_parent ORDER BY review_date
            )                                                     AS review_relative_rank,
            MIN(review_date) OVER (PARTITION BY product_parent)   AS first_review_date,
            MAX(review_date) OVER (PARTITION BY product_parent)   AS last_review_date
        FROM silver_data
    ),
    with_early AS (
        SELECT
            *,
            CASE
                WHEN review_relative_rank <= CEIL(product_review_count * 0.10)
                THEN TRUE ELSE FALSE
            END AS is_early_review,
            DATE_DIFF('day', first_review_date, review_date) AS days_since_first_review
        FROM base
    ),
    with_category_stats AS (
        SELECT
            *,
            -- Popularity percentile within category:
            -- 0.0 = least-reviewed product in category, 1.0 = most-reviewed.
            -- A product at the 90th percentile faces a far more crowded review
            -- landscape: individual reviews must work harder to stand out.
            PERCENT_RANK() OVER (
                PARTITION BY product_category_id
                ORDER BY product_review_count
            ) AS product_popularity_pctile,

            -- Category review density: average reviews per product in this category.
            -- High density = competitive category where review differentiation matters.
            AVG(product_review_count) OVER (
                PARTITION BY product_category_id
            ) AS category_review_density,

            -- Count of distinct products in this category
            COUNT(DISTINCT product_parent) OVER (
                PARTITION BY product_category_id
            ) AS category_product_count
        FROM with_early
    )
    SELECT
        _index,
        product_id,
        product_parent,
        label,
        product_category_id,
        product_review_count,
        product_popularity_pctile,
        ROUND(category_review_density, 2)  AS category_review_density,
        category_product_count,

        -- INTERACTION FEATURES
        -- Early review on a popular product: the review appeared before the product
        -- had traction, yet the product later became popular — high competitive pressure.
        (is_early_review AND product_popularity_pctile > 0.75) AS early_in_popular_product,

        -- Vine review on a product with few reviews: editorial seeding signal.
        -- Vine reviews on sparse products are disproportionately influential
        -- because they anchor early perception with no organic reviews to compete with.
        (vine = 'Y' AND product_review_count < 10)            AS vine_in_sparse_product,

        -- Multiplicative temporal × popularity interaction:
        -- Review age matters more for popular products because older reviews
        -- on popular products have had longer exposure to accumulate votes.
        -- NULL when days_since_first_review is NULL (single-date products).
        days_since_first_review * product_popularity_pctile   AS days_since_first_x_popularity,

        -- Verified purchase on a high-popularity product:
        -- On popular products with many reviews, verified purchase is a stronger
        -- differentiator since unverified reviews face higher reader scepticism.
        (verified_purchase = 'Y' AND product_popularity_pctile > 0.75) AS verified_in_popular_product,

        -- Popularity percentile x review count: absolute popularity score.
        -- Captures both relative position and absolute scale simultaneously.
        ROUND(product_popularity_pctile * product_review_count, 2) AS popularity_weight

    FROM with_category_stats
""")

print("gold.review_product_context created.")
print("Rows:", con.execute("SELECT COUNT(*) FROM gold.review_product_context").fetchone()[0])

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                                               AS total_rows,
        ROUND(AVG(product_popularity_pctile),      3)                         AS avg_popularity_pctile,
        ROUND(AVG(category_review_density),        2)                         AS avg_cat_density,
        SUM(CASE WHEN early_in_popular_product  THEN 1 ELSE 0 END)            AS early_in_popular,
        SUM(CASE WHEN vine_in_sparse_product    THEN 1 ELSE 0 END)            AS vine_sparse,
        SUM(CASE WHEN verified_in_popular_product THEN 1 ELSE 0 END)          AS verified_popular,
        ROUND(AVG(days_since_first_x_popularity),  2)                         AS avg_days_x_popularity,
        ROUND(AVG(popularity_weight),              2)                         AS avg_popularity_weight
    FROM gold.review_product_context
""").df()

In [ ]:
con.execute("""
    SELECT
        label,
        COUNT(*)                                                AS reviews,
        ROUND(AVG(product_popularity_pctile),    3)            AS avg_pop_pctile,
        ROUND(AVG(category_review_density),      2)            AS avg_cat_density,
        SUM(CASE WHEN early_in_popular_product THEN 1 ELSE 0 END)  AS early_in_popular,
        SUM(CASE WHEN vine_in_sparse_product   THEN 1 ELSE 0 END)  AS vine_sparse,
        ROUND(AVG(days_since_first_x_popularity), 2)           AS avg_days_x_pop,
        ROUND(AVG(popularity_weight),             2)           AS avg_pop_weight
    FROM gold.review_product_context
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
""").df()

In [ ]:
con.execute("""
    SELECT
        product_category_id,
        COUNT(*)                                       AS reviews,
        category_product_count,
        ROUND(category_review_density, 2)              AS avg_reviews_per_product,
        ROUND(AVG(product_popularity_pctile), 3)       AS avg_pop_pctile,
        ROUND(MAX(product_review_count), 0)            AS max_product_reviews,
        SUM(CASE WHEN early_in_popular_product THEN 1 ELSE 0 END) AS early_in_popular
    FROM gold.review_product_context
    GROUP BY product_category_id, category_product_count, category_review_density
    ORDER BY reviews DESC
""").df()

In [ ]:
gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"product_context_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_product_context)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
con.close()